In [2]:
import json
import pandas as pd

# ── Fichiers générés par ton collègue ──────────────────────────────────────
ANOMALIES_CSV      = "data/anomalies.csv"
QUASI_DOUBLONS_CSV = "data/quasi_doublons.csv"
EVENTS_CLUSTERING  = "data/events_clustering.json"   # contient cluster_id, is_noise, is_quasi_dup

# ── Chargement ─────────────────────────────────────────────────────────────
df_anom = pd.read_csv(ANOMALIES_CSV)
df_qd   = pd.read_csv(QUASI_DOUBLONS_CSV)

with open(EVENTS_CLUSTERING, encoding="utf-8") as f:
    ev_clust = json.load(f)

print(f"✅ anomalies.csv        : {len(df_anom)} lignes")
print(f"✅ quasi_doublons.csv   : {len(df_qd)} lignes")
print(f"✅ events_clustering    : {len(ev_clust)} events")
print()
print("Colonnes anomalies :", list(df_anom.columns))
print("Colonnes quasi_dup :", list(df_qd.columns))

✅ anomalies.csv        : 11948 lignes
✅ quasi_doublons.csv   : 1727 lignes
✅ events_clustering    : 11948 events

Colonnes anomalies : ['event_id', 'type', 'score_if', 'score_local', 'score_final', 'niveau', 'explication', 'context']
Colonnes quasi_dup : ['id_1', 'id_2', 'cosine', 'sim_structure', 'sij', 'type_1', 'type_2']


In [3]:
# ── Lookup anomalies : event_id -> dict ────────────────────────────────────
anom_lookup = {}
for _, row in df_anom.iterrows():
    anom_lookup[row["event_id"]] = {
        "score":       round(row["score_final"], 4),
        "niveau":      row["niveau"],          # Normal / Suspect / Critique
        "explication": row["explication"]
    }

# ── Lookup clustering : event_id -> dict ───────────────────────────────────
def get_oid(event):
    return event["_id"]["$oid"] if isinstance(event["_id"], dict) else event["_id"]

clust_lookup = {}
for e in ev_clust:
    eid = get_oid(e)
    clust_lookup[eid] = {
        "cluster_id":    e.get("cluster_id"),
        "cluster_label": e.get("cluster_label"),
        "is_noise":      e.get("is_noise", False),
        "is_quasi_dup":  e.get("is_quasi_dup", False)
    }

# ── Lookup quasi-doublons : event_id -> liste de partenaires ───────────────
dup_lookup = {}
for _, row in df_qd.iterrows():
    for id_from, id_to, type_to in [
        (row["id_1"], row["id_2"], row["type_2"]),
        (row["id_2"], row["id_1"], row["type_1"])
    ]:
        dup_lookup.setdefault(id_from, []).append({
            "duplicate_of":      id_to,
            "similarity_score":  round(row["sij"],    4),
            "cosine_similarity": round(row["cosine"], 4),
            "type_partner":      type_to
        })

print(f"✅ anom_lookup   : {len(anom_lookup)} entrées")
print(f"✅ clust_lookup  : {len(clust_lookup)} entrées")
print(f"✅ dup_lookup    : {len(dup_lookup)} entrées (events ayant au moins 1 doublon)")

✅ anom_lookup   : 11948 entrées
✅ clust_lookup  : 11948 entrées
✅ dup_lookup    : 1436 entrées (events ayant au moins 1 doublon)


In [4]:
def enrich_event(event: dict) -> dict:
    """
    Prend un event du JSON original et lui ajoute 3 blocs :
      - anomaly         : is_anomaly, score, niveau, explication
      - clustering      : cluster_id, cluster_label, is_noise, is_quasi_dup
      - quasi_duplicates: liste des events similaires (vers où il devrait pointer)
    """
    eid = get_oid(event)
    event = dict(event)  # copie superficielle pour ne pas muter l'original

    # --- ANOMALIE ---
    a = anom_lookup.get(eid, {})
    niveau = a.get("niveau", "Normal")
    event["anomaly"] = {
        "is_anomaly":  niveau in ("Suspect", "Critique"),
        "score":       a.get("score"),
        "niveau":      niveau,
        "explication": a.get("explication")
    }

    # --- CLUSTERING ---
    c = clust_lookup.get(eid, {})
    event["clustering"] = {
        "cluster_id":    c.get("cluster_id"),
        "cluster_label": c.get("cluster_label"),
        "is_noise":      c.get("is_noise"),      # True = hors cluster
        "is_quasi_dup":  c.get("is_quasi_dup", False)
    }

    # --- QUASI-DOUBLONS (vers où pointer) ---
    event["quasi_duplicates"] = dup_lookup.get(eid, [])

    return event


# ── Test rapide sur le 1er event du document ───────────────────────────────
sample_event = {
    "_id": {"$oid": "698b35ea8f03efde04a8799f"},
    "type": "Thing/Abstract/Event/Hold",
    "context": "Le procureur a tenu une conférence de presse..."
}

result = enrich_event(sample_event)
print(json.dumps({k: result[k] for k in ["anomaly", "clustering", "quasi_duplicates"]},
                 ensure_ascii=False, indent=2))

{
  "anomaly": {
    "is_anomaly": false,
    "score": 0.5317,
    "niveau": "Normal",
    "explication": "Nombre de noeuds inhabituel : 6."
  },
  "clustering": {
    "cluster_id": 595,
    "cluster_label": "presse | appel | cadre",
    "is_noise": false,
    "is_quasi_dup": true
  },
  "quasi_duplicates": [
    {
      "duplicate_of": "698b43fd8f03efde04a87ce5",
      "similarity_score": 1.0,
      "cosine_similarity": 1.0,
      "type_partner": "Thing/Abstract/Event/Hold"
    }
  ]
}


In [5]:
INPUT_FILE  = "data/export.events.json"
OUTPUT_FILE = "data/export.events.clean.json"

# Chargement
print(f"⏳ Chargement de {INPUT_FILE} ...")
with open(INPUT_FILE, encoding="utf-8") as f:
    events = json.load(f)
print(f"✅ {len(events)} events chargés")

# Enrichissement
print("⏳ Enrichissement en cours ...")
enriched = [enrich_event(e) for e in events]

# Stats
n_anomaly  = sum(1 for e in enriched if e["anomaly"]["is_anomaly"])
n_critique = sum(1 for e in enriched if e["anomaly"]["niveau"] == "Critique")
n_noise    = sum(1 for e in enriched if e["clustering"]["is_noise"])
n_dup      = sum(1 for e in enriched if e["quasi_duplicates"])

print(f"\n📊 Résumé :")
print(f"   Anomalies (Suspect + Critique) : {n_anomaly}")
print(f"   Anomalies Critique seulement   : {n_critique}")
print(f"   Events hors cluster (noise)    : {n_noise}")
print(f"   Events avec quasi-doublons     : {n_dup}")

# Sauvegarde
print(f"\n⏳ Sauvegarde dans {OUTPUT_FILE} ...")
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(enriched, f, ensure_ascii=False, indent=2)
print(f"✅ Fichier sauvegardé : {OUTPUT_FILE}")

⏳ Chargement de data/export.events.json ...
✅ 11948 events chargés
⏳ Enrichissement en cours ...

📊 Résumé :
   Anomalies (Suspect + Critique) : 2123
   Anomalies Critique seulement   : 13
   Events hors cluster (noise)    : 2875
   Events avec quasi-doublons     : 1436

⏳ Sauvegarde dans data/export.events.clean.json ...
✅ Fichier sauvegardé : data/export.events.clean.json


In [7]:
# ── 1 event Critique ───────────────────────────────────────────────────────
critique = next((e for e in enriched if e["anomaly"]["niveau"] == "Critique"), None)
if critique:
    eid = get_oid(critique)
    print(f"🔴 CRITIQUE  [{eid}]")
    print(f"   type       : {critique['type']}")
    print(f"   score      : {critique['anomaly']['score']}")
    print(f"   explication: {critique['anomaly']['explication']}")
    print(f"   cluster    : {critique['clustering']['cluster_label']}  (noise={critique['clustering']['is_noise']})")
    print(f"   doublons   : {len(critique['quasi_duplicates'])}")

print()

# ── 1 event avec quasi-doublons ────────────────────────────────────────────
with_dup = next((e for e in enriched if e["quasi_duplicates"]), None)
if with_dup:
    eid = get_oid(with_dup)
    print(f"🔁 QUASI-DUP [{eid}]")
    print(f"   type    : {with_dup['type']}")
    print(f"   anomaly : {with_dup['anomaly']['niveau']}  (is_anomaly={with_dup['anomaly']['is_anomaly']})")
    print(f"   pointe vers :")
    for d in with_dup["quasi_duplicates"]:
        print(f"     → {d['duplicate_of']}  (sij={d['similarity_score']}, type={d['type_partner']})")

print()

# ── 1 event normal sans doublon ────────────────────────────────────────────
normal = next((e for e in enriched
               if not e["anomaly"]["is_anomaly"]
               and not e["quasi_duplicates"]
               and not e["clustering"]["is_noise"]), None)
if normal:
    eid = get_oid(normal)
    print(f"✅ NORMAL    [{eid}]")
    print(f"   type    : {normal['type']}")
    print(f"   cluster : {normal['clustering']['cluster_label']}")
    print(f"   anomaly : {normal['anomaly']['niveau']}")

🔴 CRITIQUE  [698b3cf18f03efde04a87c80]
   type       : Thing/Abstract/Event/Stop
   score      : 0.8098
   explication: Contexte semantiquement isole parmi les events de type 'Thing/Abstract/Event/Stop'.
   cluster    : artiste | années | producteur  (noise=False)
   doublons   : 0

🔁 QUASI-DUP [698b35ea8f03efde04a8799f]
   type    : Thing/Abstract/Event/Hold
   anomaly : Normal  (is_anomaly=False)
   pointe vers :
     → 698b43fd8f03efde04a87ce5  (sij=1.0, type=Thing/Abstract/Event/Hold)

✅ NORMAL    [698b35e98f03efde04a8799a]
   type    : Thing/Abstract/Event/Transfer/TransferOfUnbiasedInformation
   cluster : mettent | banque | péril
   anomaly : Normal
